[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aashish7542/digit-recognition-robustness/blob/main/notebooks/robustness_experiments.ipynb)

# Robustness of handwritten digit classification to random patch noise

**Research question.** Does training with localized random pixel corruption improve an MLP classifier’s accuracy on similarly corrupted 8 × 8 handwritten digits?

This experiment uses the built-in scikit-learn `load_digits` dataset. A randomly located square patch is **replaced by uniform random pixel values**; it is not a zero-valued occlusion. Clean and augmented models share architecture and initializations for each seed.

**Protocol:** stratified 60/20/20 train/validation/test split; patch size selected on validation data; final test evaluated afterward. See the README for saved-run results and limitations.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()


image = digits.images[0]
label = digits.target[0]

plt.imshow(image, cmap="gray")
plt.title(f"Label: {label}")
plt.axis("off")
plt.show()

print("Pixel values:\n", image)
print("Image shape:", image.shape)

Pixel values:
 [[ 0.  0.  5. 13.  9.  1.  0.  0.]
 [ 0.  0. 13. 15. 10. 15.  5.  0.]
 [ 0.  3. 15.  2.  0. 11.  8.  0.]
 [ 0.  4. 12.  0.  0.  8.  8.  0.]
 [ 0.  5.  8.  0.  0.  9.  8.  0.]
 [ 0.  4. 11.  0.  1. 12.  7.  0.]
 [ 0.  2. 14.  5. 10. 12.  0.  0.]
 [ 0.  0.  6. 13. 10.  0.  0.  0.]]
Image shape: (8, 8)


In [ ]:
# Flatten each image and scale pixel values from [0, 16] to [0, 1].
X = digits.images.reshape(len(digits.images), 64) / 16.0
y = digits.target

print("Feature matrix:", X.shape)
print("Labels:", y.shape)

Feature matrix: (1797, 64)
Labels: (1797,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5, random_state=42, stratify=y_rest
)

print("Training:", len(y_train))
print("Validation:", len(y_val))
print("Test:", len(y_test))

Training: 1078
Validation: 359
Test: 360


In [ ]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(
    hidden_layer_sizes=(32,),
    activation="relu",
    solver="adam",
    batch_size=64,
    learning_rate_init=0.001,
    max_iter=200,
    random_state=42
)

model.fit(X_train, y_train)

print(f"Training accuracy:   {model.score(X_train, y_train):.1%}")
print(f"Validation accuracy: {model.score(X_val, y_val):.1%}")

Training accuracy:   100.0%
Validation accuracy: 97.5%


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
import numpy as np

val_predictions = model.predict(X_val)
mistakes = np.flatnonzero(val_predictions != y_val)

print(f"Incorrect predictions: {len(mistakes)} out of {len(y_val)}")

fig, axes = plt.subplots(1, min(9, max(len(mistakes), 1)), figsize=(12, 2))

for ax, i in zip(np.atleast_1d(axes), mistakes[:9]):
    ax.imshow(X_val[i].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"True: {y_val[i]}\nPredicted: {val_predictions[i]}")

for ax in np.atleast_1d(axes):
    ax.axis("off")

plt.tight_layout()
plt.show()

Incorrect predictions: 9 out of 359


In [ ]:
print(mistakes)
print(val_predictions[mistakes])
print(y_val[mistakes])


[  5  20  45  58  98 229 281 311 322]
[1 1 1 7 3 9 1 1 7]
[2 8 8 2 8 5 8 2 8]


## 1. Baseline error analysis

A small multilayer perceptron achieves high accuracy on clean digits. We investigate whether local corruption during training improves robustness to the same type of corruption at evaluation. This measures robustness to **patch noise**, not all image distortions.


In [ ]:
original = X_train[0].reshape(8, 8)
corrupted = original.copy()  # Preserve the original training image.

corrupted[2:4, 2:4] = 0

fig, axes = plt.subplots(1, 2, figsize=(5, 3))

for ax, img, title in zip(
    axes, [original, corrupted], ["Original", "2 × 2 patch hidden"]
):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")

plt.show()

## 2. Random patch-noise operator

Select a random square region and replace its normalized pixel values with independent samples from a uniform distribution on [0, 1]. Because the replacement pixels are random noise, this operator is *not* strict patch erasure or masking.


In [ ]:
rng = np.random.default_rng(42)

def corrupt_patch(image, rng, size=5):
    corrupted = image.copy()
    if image.ndim != 2:
        raise ValueError('Expected a 2D image')
    height, width = image.shape
    if not 1 <= size <= min(height, width):
        raise ValueError('Patch size must fit within the image')

    row = rng.integers(0, height - size + 1)
    col = rng.integers(0, width - size + 1)

    corrupted[row:row + size, col:col + size] = rng.uniform(
        0, 1, size=(size, size)
    )
    return corrupted

corrupted = corrupt_patch(original, rng)

fig, axes = plt.subplots(1, 2, figsize=(5, 3))
for ax, img, title in zip(
    axes, [original, corrupted], ["Original", "Random patch"]
):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
plt.show()

## 3. Clean-trained baseline on corrupted validation images

Evaluate the clean-trained MLP on a fixed patch-corrupted validation set. This set is generated once with a fixed seed and reused in the model comparison.


In [ ]:
# Create a reproducible patch-corrupted validation set.
val_rng = np.random.default_rng(123)

X_val_corrupted = np.array([
    corrupt_patch(row.reshape(8, 8), val_rng, size=2).ravel()
    for row in X_val
])

print(f"Clean validation:   {model.score(X_val, y_val):.1%}")
print(f"Patched validation: {model.score(X_val_corrupted, y_val):.1%}")

Clean validation:   97.5%
Patched validation: 93.0%


## 4. Sensitivity to corruption patch size

Evaluate how a clean-trained model responds to separately sampled 1 × 1 through 8 × 8 patch-noise corruptions. These samples illustrate severity and are separate from the fixed validation benchmark below.


In [ ]:
# Create a reproducible patch-corrupted validation set
val_rng = np.random.default_rng(123)

# Sweep corruption patch sizes
sizes_to_test = np.arange(1,9)

val_accuracy = []

for size in sizes_to_test:
    # Generate the corrupted validation set for this patch size
    X_val_corrupted_sweep = np.array([
        corrupt_patch(row.reshape(8, 8), val_rng, size=size).ravel()
        for row in X_val
    ])

    # Calculate and print the results for this size
    clean_score = model.score(X_val, y_val)
    patched_score = model.score(X_val_corrupted_sweep, y_val)

    print(f"--- Patch Size: {size} ---")
    print(f"Clean validation:   {clean_score:.1%}")
    print(f"Patched validation: {patched_score:.1%}\n")
    val_accuracy.append(patched_score)
plt.plot(sizes_to_test, val_accuracy, marker='o')
plt.xlabel('Patch Size')
plt.ylabel('Validation Accuracy')
plt.title('Validation Accuracy vs. Patch Size')
plt.grid(True)
plt.show()


--- Patch Size: 1 ---
Clean validation:   97.5%
Patched validation: 97.2%

--- Patch Size: 2 ---
Clean validation:   97.5%
Patched validation: 92.2%

--- Patch Size: 3 ---
Clean validation:   97.5%
Patched validation: 83.3%

--- Patch Size: 4 ---
Clean validation:   97.5%
Patched validation: 67.1%

--- Patch Size: 5 ---
Clean validation:   97.5%
Patched validation: 48.5%

--- Patch Size: 6 ---
Clean validation:   97.5%
Patched validation: 26.2%

--- Patch Size: 7 ---
Clean validation:   97.5%
Patched validation: 13.9%

--- Patch Size: 8 ---
Clean validation:   97.5%
Patched validation: 10.0%



## 5. Clean training versus noise-augmented training

Train two identical MLPs with matched initialization seeds and training budgets:

1. **Clean baseline:** unmodified training images.
2. **Patch-noise augmentation:** with probability 0.5, replace a random 2 × 2 region with uniform random pixel values before each epoch.

Noise is resampled each epoch; evaluation images stay fixed.


In [ ]:
def augment_training(X, rng, probability=0.5, size=2):
    augmented = X.copy()

    for i in range(len(X)):
        if rng.random() < probability:
            image = X[i].reshape(8, 8)
            augmented[i] = corrupt_patch(image, rng, size).ravel()

    return augmented

Use `partial_fit` for 200 epochs to resample augmentation each epoch. The models share architecture, initialization seed, number of epochs, and evaluation images; training inputs differ by design.


In [ ]:
from sklearn.base import clone

# Copy settings, but start with fresh, untrained models.
clean_model = clone(model)
augmented_model = clone(model)

train_rng = np.random.default_rng(456)
classes = np.arange(10)

for epoch in range(200):
    X_augmented = augment_training(
        X_train, train_rng, probability=0.5, size=2
    )

    clean_model.partial_fit(X_train, y_train, classes=classes)
    augmented_model.partial_fit(X_augmented, y_train, classes=classes)

for name, trained_model in [
    ("Clean training", clean_model),
    ("Noise-augmented training", augmented_model),
]:
    print(
        f"{name}: "
        f"clean accuracy = {trained_model.score(X_val, y_val):.1%}, "
        f"patched accuracy = {trained_model.score(X_val_corrupted, y_val):.1%}"
    )

Clean training: clean accuracy = 97.5%, patched accuracy = 92.8%
Noise-augmented training: clean accuracy = 96.9%, patched accuracy = 93.9%


### Variation across random initialization seeds

Repeat training with five network seeds (0–4), pairing clean and augmented runs at each seed. Report paired accuracy differences in percentage points. Five seeds provide a sensitivity check, **not** a formal statistical significance test or a guarantee of generalization.


In [ ]:
results = []

for seed in range(5):
    clean_model = clone(model).set_params(random_state=seed)
    augmented_model = clone(model).set_params(random_state=seed)
    train_rng = np.random.default_rng(1000 + seed)

    for epoch in range(200):
        X_augmented = augment_training(
            X_train, train_rng, probability=0.5, size=2
        )
        clean_model.partial_fit(X_train, y_train, classes=classes)
        augmented_model.partial_fit(X_augmented, y_train, classes=classes)

    # Compare noise-augmented training against clean training.
    clean_change = 100 * (
        augmented_model.score(X_val, y_val)
        - clean_model.score(X_val, y_val)
    )
    patched_change = 100 * (
        augmented_model.score(X_val_corrupted, y_val)
        - clean_model.score(X_val_corrupted, y_val)
    )

    results.append([clean_change, patched_change])

    print(
        f"Seed {seed}: clean change = {clean_change:+.2f} pp, " 
        f"patched change = {patched_change:+.2f} pp"
    )

Seed 0: clean change = +0.28 pp, patched change = +1.95 pp
Seed 1: clean change = +0.00 pp, patched change = +0.56 pp
Seed 2: clean change = -0.56 pp, patched change = +0.84 pp
Seed 3: clean change = +0.84 pp, patched change = +2.79 pp
Seed 4: clean change = -0.56 pp, patched change = +0.84 pp


### Validation result

In the original saved run, augmentation improved accuracy on the fixed 2 × 2 patch-noise validation set for all five seeds (gains from +0.56 to +2.79 percentage points). Clean validation accuracy fluctuated in both directions. This does not prove universal superiority.


## 6. Choosing the training corruption size

Compare 2 × 2, 4 × 4, and 6 × 6 training patches over five seeds. Measure clean accuracy and accuracy on the same fixed 2 × 2 corrupted validation set. This is a limited validation-based comparison, not an exhaustive search.


In [ ]:
patch_results = {}

for size in [2, 4, 6]:
    scores = []

    for seed in range(5):
        candidate = clone(model).set_params(random_state=seed)
        train_rng = np.random.default_rng(1000 + seed)

        for epoch in range(200):
            X_augmented = augment_training(
                X_train, train_rng, probability=0.5, size=size
            )
            candidate.partial_fit(
                X_augmented, y_train, classes=classes
            )

        scores.append([
            candidate.score(X_val, y_val),
            candidate.score(X_val_corrupted, y_val)
        ])

    patch_results[size] = np.array(scores)
    averages = patch_results[size].mean(axis=0)

    print(
        f"Training patch {size}×{size}: "
        f"mean clean accuracy = {averages[0]:.1%}, "
        f"mean patched accuracy = {averages[1]:.1%}"
    )

Training patch 2×2: mean clean accuracy = 97.1%, mean patched accuracy = 94.2%
Training patch 4×4: mean clean accuracy = 96.6%, mean patched accuracy = 94.2%
Training patch 6×6: mean clean accuracy = 94.6%, mean patched accuracy = 91.9%


### Validation-based choice

The original five-seed validation averages favored 2 × 2 training noise: **97.1% clean accuracy** and **94.2% accuracy on the fixed 2 × 2 corrupted validation set**. Larger training patches did not improve this joint trade-off. Select 2 × 2 noise at 50% probability for final testing.


## 7. Held-out test evaluation

In [ ]:
# Create the test corruption once; keep it identical for every model.
test_rng = np.random.default_rng(789)

X_test_corrupted = np.array([
    corrupt_patch(row.reshape(8, 8), test_rng, size=2).ravel()
    for row in X_test
])

test_results = []

for seed in range(5):
    clean_model = clone(model).set_params(random_state=seed)
    augmented_model = clone(model).set_params(random_state=seed)
    train_rng = np.random.default_rng(1000 + seed)

    for epoch in range(200):
        X_augmented = augment_training(
            X_train, train_rng, probability=0.5, size=2
        )
        clean_model.partial_fit(X_train, y_train, classes=classes)
        augmented_model.partial_fit(X_augmented, y_train, classes=classes)

    test_results.append([
        clean_model.score(X_test, y_test),
        clean_model.score(X_test_corrupted, y_test),
        augmented_model.score(X_test, y_test),
        augmented_model.score(X_test_corrupted, y_test)
    ])

test_results = np.array(test_results)
means = test_results.mean(axis=0)

print(f"Clean training:   clean test = {means[0]:.1%}, "
      f"patched test = {means[1]:.1%}")
print(f"Noise-augmented training: clean test = {means[2]:.1%}, "
      f"patched test = {means[3]:.1%}")

patched_gains = 100 * (test_results[:, 3] - test_results[:, 1])
print("Patched gains per seed (percentage points):",
      np.round(patched_gains, 2))

Clean training:   clean test = 96.8%, patched test = 90.7%
Noise-augmented training: clean test = 96.6%, patched test = 92.9%
Patched gains per seed (percentage points): [3.33 2.78 1.67 1.11 2.5 ]


## 8. Conclusions and limitations

The original saved run reported these held-out test averages across five network seeds:

| Training | Clean test accuracy | 2 × 2 patch-noise test accuracy |
|---|---:|---:|
| Clean only | 96.8% | 90.7% |
| 50% patch-noise augmentation | 96.6% | 92.9% |

This is a **+2.2 percentage-point** gain on the corrupted test set with a **−0.2 percentage-point** change on clean test images, for this split, model, corruption distribution, and training budget.

**Limitations.** The dataset contains low-resolution 8 × 8 digits; only one MLP architecture is tested; the test set is modest; and five seeds do not establish statistical significance. Random patch noise is not equivalent to real occlusion, blur, lighting changes, or adversarial perturbations. The fixed corrupted test images support paired comparisons, not generalization across corruption patterns. Future work should test additional corruptions, larger datasets, and uncertainty estimates.
